In [1]:
import json
import pandas as pd
import numpy as np
import requests_cache
from retry_requests import retry
import openmeteo_requests

In [2]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [3]:
# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://marine-api.open-meteo.com/v1/marine"
params = {
	"latitude": [52.4693, 52.1066],
	"longitude": [4.556, 4.2654],
	"hourly": ["wave_height", "wave_direction", "wave_period", "wind_wave_peak_period", "wind_wave_height", "wind_wave_direction", "wind_wave_period", "swell_wave_height", "swell_wave_period", "swell_wave_direction", "swell_wave_peak_period", "secondary_swell_wave_height", "secondary_swell_wave_period", "secondary_swell_wave_direction", "sea_surface_temperature"],
}
responses = openmeteo.weather_api(url, params=params)

In [4]:
locations = ["wijk_aan_zee_noordpier", "scheveningen"]

In [5]:
# empty dict for dataframes
dfs = {}

# Process locations
for response, location in zip(responses, locations):

	# Process hourly data. The order of variables needs to be the same as requested.
	hourly = response.Hourly()
	hourly_wave_height = hourly.Variables(0).ValuesAsNumpy()
	hourly_wave_direction = hourly.Variables(1).ValuesAsNumpy()
	hourly_wave_period = hourly.Variables(2).ValuesAsNumpy()
	hourly_wind_wave_peak_period = hourly.Variables(3).ValuesAsNumpy()
	hourly_wind_wave_height = hourly.Variables(4).ValuesAsNumpy()
	hourly_wind_wave_direction = hourly.Variables(5).ValuesAsNumpy()
	hourly_wind_wave_period = hourly.Variables(6).ValuesAsNumpy()
	hourly_swell_wave_height = hourly.Variables(7).ValuesAsNumpy()
	hourly_swell_wave_period = hourly.Variables(8).ValuesAsNumpy()
	hourly_swell_wave_direction = hourly.Variables(9).ValuesAsNumpy()
	hourly_swell_wave_peak_period = hourly.Variables(10).ValuesAsNumpy()
	hourly_secondary_swell_wave_height = hourly.Variables(11).ValuesAsNumpy()
	hourly_secondary_swell_wave_period = hourly.Variables(12).ValuesAsNumpy()
	hourly_secondary_swell_wave_direction = hourly.Variables(13).ValuesAsNumpy()
	hourly_sea_surface_temperature = hourly.Variables(14).ValuesAsNumpy()
	
	hourly_data = {"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)}
	
	hourly_data["wave_height"] = hourly_wave_height
	hourly_data["wave_direction"] = hourly_wave_direction
	hourly_data["wave_period"] = hourly_wave_period
	hourly_data["wind_wave_peak_period"] = hourly_wind_wave_peak_period
	hourly_data["wind_wave_height"] = hourly_wind_wave_height
	hourly_data["wind_wave_direction"] = hourly_wind_wave_direction
	hourly_data["wind_wave_period"] = hourly_wind_wave_period
	hourly_data["swell_wave_height"] = hourly_swell_wave_height
	hourly_data["swell_wave_period"] = hourly_swell_wave_period
	hourly_data["swell_wave_direction"] = hourly_swell_wave_direction
	hourly_data["swell_wave_peak_period"] = hourly_swell_wave_peak_period
	hourly_data["secondary_swell_wave_height"] = hourly_secondary_swell_wave_height
	hourly_data["secondary_swell_wave_period"] = hourly_secondary_swell_wave_period
	hourly_data["secondary_swell_wave_direction"] = hourly_secondary_swell_wave_direction
	hourly_data["sea_surface_temperature"] = hourly_sea_surface_temperature
	
	hourly_dataframe = pd.DataFrame(data = hourly_data)
	dfs[location] = hourly_dataframe

In [6]:
# aannames:
# - dfs: dict[str -> pd.DataFrame] met kolom 'date' (tz-aware of UTC epoch naar dt al gedaan)
# - alle overige kolommen zoals in je voorbeeld aanwezig

TZ = "Europe/Amsterdam"
DUTCH_DAY = ["Maandag","Dinsdag","Woensdag","Donderdag","Vrijdag","Zaterdag","Zondag"]

# Exacte doeldvolgorde (en labels) zoals jij vroeg
FIELD_ORDER = [
    "Datum", "Dag", "Locatie", "Tijd",
    "wave_height", "wave_direction", "wave_period",
    "wind_wave_peak_period", "wind_wave_height", "wind_wave_direction", "wind_wave_period",
    "swell_wave_height", "swell_wave_period", "swell_wave_direction", "swell_wave_peak_period",
    "secondary_swell_wave_height", "secondary_swell_wave_period", "secondary_swell_wave_direction",
    "sea_surface_temperature",
]

def to_json_array(dfs: dict[str, pd.DataFrame]) -> str:
    out = []

    for location, df in dfs.items():
        df = df.copy()

        # Zorg dat 'date' een tz-aware datetime is, en converteer naar lokale tijd
        df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
        df["date_local"] = df["date"].dt.tz_convert(TZ)

        # Afgeleiden voor de vereiste velden
        df["Datum"] = df["date_local"].dt.strftime("%Y-%m-%d")
        df["Dag"] = df["date_local"].dt.weekday.map(lambda i: DUTCH_DAY[i])
        df["Locatie"] = location
        df["Tijd"] = df["date_local"].dt.strftime("%H:00")  # uur van de dag

        # Maak NaN -> None voor correcte JSON nulls
        df = df.replace({np.nan: None})

        # Zet rijen om naar dicts in de exacte veldvolgorde
        for _, row in df.iterrows():
            record = {
                "Datum": row["Datum"],
                "Dag": row["Dag"],
                "Locatie": row["Locatie"],
                "Tijd": row["Tijd"],
                "wave_height": row.get("wave_height"),
                "wave_direction": row.get("wave_direction"),
                "wave_period": row.get("wave_period"),
                "wind_wave_peak_period": row.get("wind_wave_peak_period"),
                "wind_wave_height": row.get("wind_wave_height"),
                "wind_wave_direction": row.get("wind_wave_direction"),
                "wind_wave_period": row.get("wind_wave_period"),
                "swell_wave_height": row.get("swell_wave_height"),
                "swell_wave_period": row.get("swell_wave_period"),
                "swell_wave_direction": row.get("swell_wave_direction"),
                "swell_wave_peak_period": row.get("swell_wave_peak_period"),
                "secondary_swell_wave_height": row.get("secondary_swell_wave_height"),
                "secondary_swell_wave_period": row.get("secondary_swell_wave_period"),
                "secondary_swell_wave_direction": row.get("secondary_swell_wave_direction"),
                "sea_surface_temperature": row.get("sea_surface_temperature"),
            }
            # Bewaak exacte volgorde door FIELD_ORDER
            record = {k: record[k] for k in FIELD_ORDER}
            out.append(record)

    # Unicode (NL dag-namen) behouden, NaN al vervangen, dus standaard is oké
    return json.dumps(out, ensure_ascii=False)


In [7]:

# Voorbeeld gebruik:
json_array_str = to_json_array(dfs)
print(json_array_str[:1000])  # inspecteer een stukje

[{"Datum": "2025-09-11", "Dag": "Donderdag", "Locatie": "wijk_aan_zee_noordpier", "Tijd": "02:00", "wave_height": 0.7400000095367432, "wave_direction": 238.0, "wave_period": 4.25, "wind_wave_peak_period": 2.450000047683716, "wind_wave_height": 0.36000001430511475, "wind_wave_direction": 189.0, "wind_wave_period": 2.4000000953674316, "swell_wave_height": 0.6399999856948853, "swell_wave_period": 4.800000190734863, "swell_wave_direction": 249.0, "swell_wave_peak_period": 5.199999809265137, "secondary_swell_wave_height": null, "secondary_swell_wave_period": null, "secondary_swell_wave_direction": null, "sea_surface_temperature": 19.049999237060547}, {"Datum": "2025-09-11", "Dag": "Donderdag", "Locatie": "wijk_aan_zee_noordpier", "Tijd": "03:00", "wave_height": 0.8600000143051147, "wave_direction": 237.0, "wave_period": 4.449999809265137, "wind_wave_peak_period": 3.9000000953674316, "wind_wave_height": 0.5, "wind_wave_direction": 198.0, "wind_wave_period": 2.8499999046325684, "swell_wave_he